In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

import sys
import os

# Add the library path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

from ex_fuzzy_reg.rules_reg_utils import generate_triangular_partitions, generate_trapezoidal_partitions
from ex_fuzzy_reg.evolutionary_fit_reg import BaseFuzzyRulesRegressor

np.set_printoptions(suppress=True)

In [2]:
# TV,Radio,Newspaper,Sales
advertising_data = np.loadtxt('datasets/advertising.csv', delimiter=',', skiprows=1)
label_names = ["TV", "Radio", "Newspaper", "Sales"]

X = advertising_data[:, :-1]
y = advertising_data[:, -1].reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=12)

print("First 10 samples:")
print(np.hstack((X_test[:10], y_test[:10])))

First 10 samples:
[[228.3  16.9  26.2  20.5]
 [ 56.2   5.7  29.7   8.7]
 [ 19.6  20.1  17.    7.6]
 [237.4   5.1  23.5  17.5]
 [102.7  29.6   8.4  14. ]
 [214.7  24.    4.   17.4]
 [261.3  42.7  54.7  24.2]
 [ 97.2   1.5  30.   13.2]
 [117.2  14.7   5.4  11.9]
 [120.2  19.6  11.6  13.2]]


In [3]:
fuzzy_variables = generate_triangular_partitions(advertising_data, n_labels=3, fv_label_names=label_names)

antecedents = fuzzy_variables[:-1]
consequent = fuzzy_variables[-1]

bfrr = BaseFuzzyRulesRegressor(n_rules=30, fuzzy_set_type='triangular', antecedents=antecedents, consequent=consequent, n_ants=3, verbose=False, optimize_lv=False)
bfrr.fit(X_train, y_train, random_state=13)

y_pred = bfrr.predict(X_test)

MSE = mean_squared_error(y_test, y_pred)
MAE = mean_absolute_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {MSE}")
print(f"Mean Absolute Error: {MAE}")
print(f"R² Score: {R2}")
print(f"Root Mean Squared Error: {RMSE}")

PyMoo backend using CPU
Mean Squared Error: 8.654611529111763
Mean Absolute Error: 2.530210808334024
R² Score: 0.7155732982124594
Root Mean Squared Error: 2.9418721129770007


In [4]:
fuzzy_variables = generate_triangular_partitions(advertising_data, n_labels=3, fv_label_names=label_names)

antecedents = fuzzy_variables[:-1]
consequent = fuzzy_variables[-1]

bfrr = BaseFuzzyRulesRegressor(n_rules=30, fuzzy_set_type='triangular', antecedents=antecedents, consequent=consequent, n_ants=3, verbose=False, optimize_lv=False, backend='evox')
bfrr.fit(X_train, y_train, random_state=13)

y_pred = bfrr.predict(X_test)

MSE = mean_squared_error(y_test, y_pred)
MAE = mean_absolute_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {MSE}")
print(f"Mean Absolute Error: {MAE}")
print(f"R² Score: {R2}")
print(f"Root Mean Squared Error: {RMSE}")

EvoX backend using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
Mean Squared Error: 6.515512518280947
Mean Absolute Error: 1.9771617477331114
R² Score: 0.7858730308348942
Root Mean Squared Error: 2.552550198973753


In [6]:
fuzzy_variables = generate_triangular_partitions(advertising_data, n_labels=3, fv_label_names=label_names)

antecedents = fuzzy_variables[:-1]
consequent = fuzzy_variables[-1]

results = {}
for seed in [33, 42, 7, 99, 123]:
    for backend in ['pymoo', 'evox']:
        bfrr = BaseFuzzyRulesRegressor(
            n_rules=30, fuzzy_set_type='triangular',
            antecedents=antecedents, consequent=consequent,
            n_ants=3, verbose=False, optimize_lv=False,
            backend=backend
        )
        bfrr.fit(X_train, y_train, random_state=seed, mut_prob=1.0)

        y_pred = bfrr.predict(X_test)
        results[(backend, seed)] = r2_score(y_test, y_pred)
        print(f"{backend} seed={seed}: R²={results[(backend, seed)]:.4f}")

PyMoo backend using CPU
pymoo seed=33: R²=0.8718
EvoX backend using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
evox seed=33: R²=0.9075
PyMoo backend using CPU
pymoo seed=42: R²=0.8159
EvoX backend using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
evox seed=42: R²=0.9162
PyMoo backend using CPU
pymoo seed=7: R²=0.8654
EvoX backend using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
evox seed=7: R²=0.8433
PyMoo backend using CPU
pymoo seed=99: R²=0.8216
EvoX backend using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
evox seed=99: R²=0.8921
PyMoo backend using CPU
pymoo seed=123: R²=0.8153
EvoX backend using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
evox seed=123: R²=0.8930


In [5]:
import time

for backend in ['pymoo', 'evox']:
    bfrr = BaseFuzzyRulesRegressor(
        n_rules=30, fuzzy_set_type='triangular',
        antecedents=antecedents, consequent=consequent,
        n_ants=3, verbose=False, optimize_lv=False,
        backend=backend
    )
    start = time.time()
    bfrr.fit(X_train, y_train, random_state=33)
    elapsed = time.time() - start
    print(f"{backend}: {elapsed:.2f}s")

PyMoo backend using CPU
pymoo: 4.39s
EvoX backend using GPU: NVIDIA GeForce RTX 3050 Laptop GPU
evox: 0.23s


### Different models to compare the results:

In [ ]:
# TODO: add brief explanation for the models used
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor

- #### Linear Regression:

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_pred = lin_reg.predict(X_test)

MSE = mean_squared_error(y_test, y_pred)
MAE = mean_absolute_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {MSE}")
print(f"Mean Absolute Error: {MAE}")
print(f"R² Score: {R2}")
print(f"Root Mean Squared Error: {RMSE}")

Mean Squared Error: 1.9474231382316722
Mean Absolute Error: 1.0410039192230955
R² Score: 0.93599953754957
Root Mean Squared Error: 1.3955010348371915


- #### K-Nearest-Neighbors:

In [ ]:
knn = KNeighborsRegressor()
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

MSE = mean_squared_error(y_test, y_pred)
MAE = mean_absolute_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {MSE}")
print(f"Mean Absolute Error: {MAE}")
print(f"R² Score: {R2}")
print(f"Root Mean Squared Error: {RMSE}")

Mean Squared Error: 1.5120266666666669
Mean Absolute Error: 0.9546666666666666
R² Score: 0.9503084850928084
Root Mean Squared Error: 1.229644935201486


### Comparing BaseFuzzyRulesRegressor in different datasets:

- #### Diabetes dataset:

In [ ]:
# TODO: add brief explanation for the datasets used
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(scaled=True)

X, y = diabetes.data, diabetes.target
print(X.shape, y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=12)

(442, 10) (442,)


In [ ]:
diabetes_data = np.concatenate((X, y.reshape(-1, 1)), axis=1)
#fuzzy_variables = generate_triangular_partitions(diabetes_data, n_labels=3)
fuzzy_variables = generate_trapezoidal_partitions(diabetes_data, n_labels=3)

antecedents = fuzzy_variables[:-1]
consequent = fuzzy_variables[-1]

print(diabetes_data.shape[1])
print(len(antecedents))

#bfrr = BaseFuzzyRulesRegressor(n_rules=35, fuzzy_set_type='trapezoidal', antecedents=antecedents, consequent=consequent, n_ants=10, verbose=True, n_linguistic_variables=4)
#bfrr = BaseFuzzyRulesRegressor(n_rules=35, fuzzy_set_type='trapezoidal', antecedents=antecedents, consequent=consequent, n_ants=10, verbose=True, n_linguistic_variables=4, backend='evox')
#bfrr = BaseFuzzyRulesRegressor(n_rules=35, fuzzy_set_type='triangular', antecedents=antecedents, consequent=consequent, n_ants=10, verbose=True, n_linguistic_variables=3)
#bfrr = BaseFuzzyRulesRegressor(n_rules=30, fuzzy_set_type='trapezoidal', antecedents=antecedents, consequent=consequent, n_ants=3, verbose=True, backend='evox')
bfrr = BaseFuzzyRulesRegressor(n_rules=30, fuzzy_set_type='trapezoidal', antecedents=antecedents, consequent=consequent, n_linguistic_variables=X.shape[1], n_ants=10, verbose=False)
#bfrr = BaseFuzzyRulesRegressor(n_rules=30, fuzzy_set_type='trapezoidal', antecedents=antecedents, consequent=consequent, n_linguistic_variables=X.shape[1], n_ants=10, verbose=False, backend='evox')
#bfrr = BaseFuzzyRulesRegressor(fuzzy_set_type='triangular', n_ants=10, verbose=True, n_linguistic_variables=3, optimize_lv=True, backend='evox')
bfrr.fit(X_train, y_train)

y_pred = bfrr.predict(X_test)

MSE = mean_squared_error(y_test, y_pred)
MAE = mean_absolute_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {MSE}")
print(f"Mean Absolute Error: {MAE}")
print(f"R² Score: {R2}")
print(f"Root Mean Squared Error: {RMSE}")

11
10
PyMoo backend using CPU
Mean Squared Error: 5276.921561112458
Mean Absolute Error: 62.56528061904746
R² Score: 0.06697377384128955
Root Mean Squared Error: 72.64242259941815


In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

y_pred = lin_reg.predict(X_test)

MSE = mean_squared_error(y_test, y_pred)
MAE = mean_absolute_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {MSE}")
print(f"Mean Absolute Error: {MAE}")
print(f"R² Score: {R2}")
print(f"Root Mean Squared Error: {RMSE}")

Mean Squared Error: 3111.314086480235
Mean Absolute Error: 46.03688831832304
R² Score: 0.44988046403875215
Root Mean Squared Error: 55.77915458735669


In [ ]:
knn = KNeighborsRegressor()
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

MSE = mean_squared_error(y_test, y_pred)
MAE = mean_absolute_error(y_test, y_pred)
R2 = r2_score(y_test, y_pred)
RMSE = root_mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {MSE}")
print(f"Mean Absolute Error: {MAE}")
print(f"R² Score: {R2}")
print(f"Root Mean Squared Error: {RMSE}")

Mean Squared Error: 4212.957014925373
Mean Absolute Error: 50.96417910447761
R² Score: 0.25509611255695597
Root Mean Squared Error: 64.90729554468722


- #### California housing dataset:

In [ ]:
from sklearn.datasets import fetch_california_housing
X, y = fetch_california_housing(download_if_missing=True, return_X_y=True)